
# Amazon Search — Week 7 V1.3
## Latency Optimization — Full 10K Catalog + Real LambdaMART

This notebook keeps the **same correct V1.1 system** and optimizes runtime only.

### Main optimizations
- Use `CLIPTextModelWithProjection` instead of loading the full CLIP vision+text model for query-time text encoding.
- Precompute catalog text normalization, token lists, lengths, metadata arrays, and CLIP row mappings once at startup.
- Cache SBERT and CLIP query embeddings.
- Reduce repeated pandas/string operations inside Top-100 feature generation.
- Measure **fresh-query** and **cached-query** latency separately.

### Upload the same 5 ZIP files
1. `amazon_week1_v4_processed_data*.zip`
2. `amazon_week4_v1_sbert_faiss_results*.zip`
3. `amazon_week4_v2_clip_multimodal_results*.zip`
4. `amazon_week5_v1_feature_dataset.zip`
5. `amazon_week5_v2_lambdamart_results.zip`

Pipeline:

`Query → SBERT → FAISS Top-100 → 29 Week5 features → saved LambdaMART → Top-10`


In [ ]:

# 1. Install dependencies
!pip -q install pandas pyarrow numpy scikit-learn faiss-cpu lightgbm sentence-transformers rank-bm25 transformers accelerate

print("✅ Dependencies installed")


In [ ]:

# 2. Upload and extract the five required archives
from google.colab import files
from pathlib import Path
from functools import lru_cache
import zipfile, shutil, json, re, warnings, time, os
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path("/content/amazon_week7_v1_3")
if ROOT.exists():
    shutil.rmtree(ROOT)

UPLOAD = ROOT / "uploads"
EXTRACT = ROOT / "extracted"
OUT = ROOT / "outputs"

for d in [UPLOAD, EXTRACT, OUT]:
    d.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for name, payload in uploaded.items():
    (UPLOAD / name).write_bytes(payload)

def pick_zip(tokens):
    for p in UPLOAD.glob("*.zip"):
        low = p.name.lower()
        if all(t.lower() in low for t in tokens):
            return p
    return None

archives = {
    "week1": pick_zip(["week1", "processed"]),
    "week4_v1": pick_zip(["week4", "v1", "sbert"]),
    "week4_v2": pick_zip(["week4", "v2", "clip"]),
    "week5_v1": pick_zip(["week5", "v1", "feature"]),
    "week5_v2": pick_zip(["week5", "v2", "lambda"]),
}

missing = [k for k,v in archives.items() if v is None]
if missing:
    print("Detected:")
    for k,v in archives.items():
        print(f"  {k}: {v}")
    raise ValueError("Missing required ZIP(s): " + ", ".join(missing))

for key, zp in archives.items():
    dest = EXTRACT / key
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zp, "r") as z:
        z.extractall(dest)
    print(f"✅ {key}: {zp.name}")

def must_find(folder, filename):
    hits = list(folder.rglob(filename))
    if not hits:
        hits = [
            p for p in folder.rglob("*")
            if p.is_file() and p.name.lower() == filename.lower()
        ]
    if not hits:
        raise FileNotFoundError(f"{filename} not found under {folder}")
    return hits[0]

W1 = EXTRACT / "week1"
W4V1 = EXTRACT / "week4_v1"
W4V2 = EXTRACT / "week4_v2"
W5V1 = EXTRACT / "week5_v1"
W5V2 = EXTRACT / "week5_v2"


In [ ]:

# 3. Load artifacts and verify exact alignment
catalog = pd.read_parquet(
    must_find(W1, "cleaned_products_10k.parquet")
).reset_index(drop=True)

for c in ["product_id", "title", "description", "category", "brand"]:
    if c not in catalog.columns:
        catalog[c] = ""
    catalog[c] = catalog[c].fillna("").astype(str)

for c in ["price", "average_rating", "review_count"]:
    if c not in catalog.columns:
        catalog[c] = np.nan
    catalog[c] = pd.to_numeric(catalog[c], errors="coerce")

catalog["product_id"] = catalog["product_id"].astype(str)

sbert_product_emb = np.load(
    must_find(W4V1, "week4_sbert_product_embeddings.npy")
).astype(np.float32)

if len(sbert_product_emb) != len(catalog):
    raise ValueError(
        f"SBERT/catalog mismatch: {len(sbert_product_emb)} vs {len(catalog)}"
    )

sbert_product_emb /= np.clip(
    np.linalg.norm(sbert_product_emb, axis=1, keepdims=True),
    1e-12,
    None
)

clip_img_emb = np.load(
    must_find(W4V2, "week4_clip_image_embeddings.npy")
).astype(np.float32)

clip_map = pd.read_csv(
    must_find(W4V2, "week4_clip_embedding_map.csv")
)
clip_map["product_id"] = clip_map["product_id"].astype(str)

if "clip_row" not in clip_map.columns:
    clip_map["clip_row"] = np.arange(len(clip_map))

if len(clip_img_emb) != len(clip_map):
    raise ValueError(
        f"CLIP embedding/map mismatch: {len(clip_img_emb)} vs {len(clip_map)}"
    )

clip_img_emb /= np.clip(
    np.linalg.norm(clip_img_emb, axis=1, keepdims=True),
    1e-12,
    None
)

w5_dataset = pd.read_parquet(
    must_find(W5V1, "week5_feature_dataset.parquet")
)

with open(must_find(W5V1, "week5_feature_list.json"), "r") as f:
    feature_cols_v1 = json.load(f)

with open(must_find(W5V2, "week5_v2_config.json"), "r") as f:
    w5_config = json.load(f)

feature_cols = w5_config.get("feature_cols", feature_cols_v1)
model_path = must_find(W5V2, "week5_v2_lambdamart_model.txt")

print(f"✅ Catalog rows: {len(catalog):,}")
print(f"✅ SBERT: {sbert_product_emb.shape}")
print(f"✅ CLIP: {clip_img_emb.shape}")
print(f"✅ Saved LambdaMART feature count: {len(feature_cols)}")


In [ ]:

# 4. Load query-time models
import faiss
import lightgbm as lgb
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, CLIPTextModelWithProjection
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SBERT_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CLIP_NAME = "openai/clip-vit-base-patch32"

# SBERT
sbert_model = SentenceTransformer(SBERT_NAME)

# FAISS over the Week4 product embeddings
faiss_index = faiss.IndexFlatIP(sbert_product_emb.shape[1])
faiss_index.add(sbert_product_emb.astype(np.float32))

# Saved Week5 LambdaMART
ranker = lgb.Booster(model_file=str(model_path))
if ranker.num_feature() != len(feature_cols):
    raise ValueError(
        f"Model expects {ranker.num_feature()} features but schema has {len(feature_cols)}"
    )

# Optimization: text tower + projection only; no CLIP vision tower at query time.
clip_device = "cuda" if torch.cuda.is_available() else "cpu"
clip_tokenizer = AutoTokenizer.from_pretrained(CLIP_NAME)
clip_text_model = CLIPTextModelWithProjection.from_pretrained(CLIP_NAME).to(clip_device)
clip_text_model.eval()

print("✅ SBERT ready")
print("✅ FAISS ntotal:", faiss_index.ntotal)
print("✅ LambdaMART features:", ranker.num_feature())
print("✅ CLIP text-only tower device:", clip_device)


In [ ]:

# 5. Precompute immutable catalog-side runtime features
TOKEN_RE = re.compile(r"[a-z0-9]+")

def tokenize(text):
    return TOKEN_RE.findall(str(text).lower())

def norm_text(x):
    x = "" if pd.isna(x) else str(x).lower()
    x = re.sub(r"[^a-z0-9\s]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()

N = len(catalog)

# Raw arrays
PID = catalog["product_id"].astype(str).to_numpy()
TITLE = catalog["title"].astype(str).to_numpy()
DESC = catalog["description"].astype(str).to_numpy()
CATEGORY = catalog["category"].astype(str).to_numpy()
BRAND = catalog["brand"].astype(str).to_numpy()
DOCUMENT = np.char.add(np.char.add(TITLE.astype(str), " "), DESC.astype(str))

RAW_PRICE = pd.to_numeric(catalog["price"], errors="coerce").to_numpy(dtype=np.float32)
RAW_RATING = pd.to_numeric(catalog["average_rating"], errors="coerce").to_numpy(dtype=np.float32)
RAW_REVIEWS = pd.to_numeric(catalog["review_count"], errors="coerce").to_numpy(dtype=np.float32)

# Normalized strings
TITLE_NORM = np.array([norm_text(x) for x in TITLE], dtype=object)
BRAND_NORM = np.array([norm_text(x) for x in BRAND], dtype=object)
CATEGORY_NORM = np.array([norm_text(x) for x in CATEGORY], dtype=object)

# Tokenized catalog fields
TITLE_TOKENS = [tokenize(x) for x in TITLE]
DESC_TOKENS = [tokenize(x) for x in DESC]
DOC_TOKENS = [tokenize(x) for x in DOCUMENT]

TITLE_TOKEN_SETS = [set(x) for x in TITLE_TOKENS]
DESC_TOKEN_SETS = [set(x) for x in DESC_TOKENS]
CATEGORY_TOKEN_SETS = [set(tokenize(x)) for x in CATEGORY]

TITLE_LEN = np.array([len(x) for x in TITLE_TOKENS], dtype=np.float32)
DESC_LEN = np.array([len(x) for x in DESC_TOKENS], dtype=np.float32)

# Direct product -> CLIP embedding row array avoids dict lookup in every query.
clip_row_by_catalog = np.full(N, -1, dtype=np.int32)
pid_to_catalog = {pid: i for i, pid in enumerate(PID)}

for pid, row in zip(
    clip_map["product_id"].astype(str),
    clip_map["clip_row"].astype(int)
):
    i = pid_to_catalog.get(pid)
    if i is not None:
        clip_row_by_catalog[i] = int(row)

# Training-side medians used for the saved model's metadata features.
train_w5 = w5_dataset[w5_dataset["split"].astype(str).eq("train")]

train_medians = {}
for c in ["price", "average_rating", "review_count"]:
    med = pd.to_numeric(train_w5[c], errors="coerce").median() if c in train_w5.columns else 0.0
    train_medians[c] = 0.0 if pd.isna(med) else float(med)

print("✅ Catalog-side preprocessing complete")
print("✅ Training medians:", train_medians)


In [ ]:

# 6. Cached query encoders
@lru_cache(maxsize=512)
def cached_sbert_query(query):
    emb = sbert_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)[0]
    return emb

@lru_cache(maxsize=512)
def cached_clip_query(query):
    batch = clip_tokenizer(
        [query],
        padding=True,
        truncation=True,
        max_length=77,
        return_tensors="pt"
    ).to(clip_device)

    with torch.inference_mode():
        out = clip_text_model(**batch)
        feat = out.text_embeds
        feat = feat / feat.norm(
            p=2, dim=-1, keepdim=True
        ).clamp_min(1e-12)

    return feat.detach().cpu().numpy().astype(np.float32)[0]

def clear_query_caches():
    cached_sbert_query.cache_clear()
    cached_clip_query.cache_clear()

print("✅ Query embedding caches ready")


In [ ]:

# 7. Optimized Top-100 feature generation
def minmax_array(x):
    x = np.asarray(x, dtype=np.float32)
    if len(x) == 0:
        return x
    lo = float(np.nanmin(x))
    hi = float(np.nanmax(x))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros(len(x), dtype=np.float32)
    return ((x - lo) / (hi - lo)).astype(np.float32)

def optimized_runtime_features(query, candidate_idx, sbert_scores):
    idx = np.asarray(candidate_idx, dtype=np.int64)
    n = len(idx)

    q_tokens = tokenize(query)
    q_set = set(q_tokens)
    q_norm = norm_text(query)

    docs = [DOCUMENT[i] for i in idx]

    # BM25 reuses pre-tokenized catalog docs.
    tokenized_docs = [DOC_TOKENS[i] for i in idx]
    bm25 = BM25Okapi(tokenized_docs)
    bm25_score = np.asarray(
        bm25.get_scores(q_tokens),
        dtype=np.float32
    )

    # Preserve Week2/Week5 candidate-local TF-IDF semantics.
    vec = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1,2),
        min_df=1
    )
    Xtf = vec.fit_transform([query] + docs)
    tfidf_similarity = cosine_similarity(
        Xtf[0:1], Xtf[1:]
    ).ravel().astype(np.float32)

    # Metadata by NumPy indexing.
    raw_price = RAW_PRICE[idx]
    raw_rating = RAW_RATING[idx]
    raw_reviews = RAW_REVIEWS[idx]

    price_missing = np.isnan(raw_price).astype(np.float32)
    rating_missing = np.isnan(raw_rating).astype(np.float32)

    price = np.where(
        np.isnan(raw_price),
        train_medians["price"],
        raw_price
    ).astype(np.float32)

    rating = np.where(
        np.isnan(raw_rating),
        train_medians["average_rating"],
        raw_rating
    ).astype(np.float32)

    reviews = np.where(
        np.isnan(raw_reviews),
        train_medians["review_count"],
        raw_reviews
    ).astype(np.float32)
    reviews = np.clip(reviews, 0, None)

    query_len = np.full(n, len(q_tokens), dtype=np.float32)
    title_len = TITLE_LEN[idx]
    description_len = DESC_LEN[idx]

    if q_set:
        qn = float(len(q_set))
        query_title_overlap = np.array(
            [len(q_set & TITLE_TOKEN_SETS[i]) / qn for i in idx],
            dtype=np.float32
        )
        query_description_overlap = np.array(
            [len(q_set & DESC_TOKEN_SETS[i]) / qn for i in idx],
            dtype=np.float32
        )
        query_title_token_coverage = query_title_overlap.copy()
        category_token_match = np.array(
            [len(q_set & CATEGORY_TOKEN_SETS[i]) / qn for i in idx],
            dtype=np.float32
        )
    else:
        query_title_overlap = np.zeros(n, dtype=np.float32)
        query_description_overlap = np.zeros(n, dtype=np.float32)
        query_title_token_coverage = np.zeros(n, dtype=np.float32)
        category_token_match = np.zeros(n, dtype=np.float32)

    sbert_similarity = np.asarray(sbert_scores, dtype=np.float32)
    sbert_available = np.ones(n, dtype=np.float32)

    # One cached CLIP text embedding; matrix multiply against available image vectors.
    q_clip = cached_clip_query(query)
    rows = clip_row_by_catalog[idx]
    valid_clip = rows >= 0

    clip_similarity = np.zeros(n, dtype=np.float32)
    clip_available = valid_clip.astype(np.float32)

    if valid_clip.any():
        clip_similarity[valid_clip] = (
            clip_img_emb[rows[valid_clip]] @ q_clip
        ).astype(np.float32)

    selected_title_norm = TITLE_NORM[idx]
    selected_brand_norm = BRAND_NORM[idx]

    exact_title_match = np.fromiter(
        (float(t == q_norm) for t in selected_title_norm),
        dtype=np.float32,
        count=n
    )

    query_in_title = np.fromiter(
        (float(bool(q_norm) and q_norm in t) for t in selected_title_norm),
        dtype=np.float32,
        count=n
    )

    brand_in_query = np.fromiter(
        (float(bool(b) and b in q_norm) for b in selected_brand_norm),
        dtype=np.float32,
        count=n
    )

    log_review_count = np.log1p(reviews).astype(np.float32)
    log_price = np.log1p(np.clip(price, 0, None)).astype(np.float32)

    data = {
        "bm25_score": bm25_score,
        "tfidf_similarity": tfidf_similarity,
        "price": price,
        "average_rating": rating,
        "review_count": reviews,
        "log_review_count": log_review_count,
        "price_missing": price_missing,
        "query_len": query_len,
        "title_len": title_len,
        "description_len": description_len,
        "query_title_overlap": query_title_overlap,
        "query_description_overlap": query_description_overlap,
        "sbert_similarity": sbert_similarity,
        "clip_similarity": clip_similarity,
        "sbert_available": sbert_available,
        "clip_available": clip_available,
        "log_price": log_price,
        "log_review_count_w5": log_review_count,
        "rating_missing_w5": rating_missing,
        "price_missing_w5": price_missing,
        "exact_title_match": exact_title_match,
        "query_in_title": query_in_title,
        "query_title_token_coverage": query_title_token_coverage,
        "brand_in_query": brand_in_query,
        "category_token_match": category_token_match,
        "bm25_score_qnorm": minmax_array(bm25_score),
        "tfidf_similarity_qnorm": minmax_array(tfidf_similarity),
        "sbert_similarity_qnorm": minmax_array(sbert_similarity),
        "clip_similarity_qnorm": minmax_array(clip_similarity),
    }

    X = pd.DataFrame(data)

    missing = [c for c in feature_cols if c not in X.columns]
    if missing:
        raise ValueError("Unimplemented saved-model features: " + ", ".join(missing))

    X = X[feature_cols].astype(np.float32)
    X = X.replace([np.inf, -np.inf], 0).fillna(0)

    return X


In [ ]:

# 8. Optimized end-to-end search
from dataclasses import dataclass, asdict

@dataclass
class SearchLatency:
    sbert_embedding_ms: float
    faiss_retrieval_ms: float
    feature_generation_ms: float
    lambdamart_ms: float
    total_ms: float

def search_products(query, top_k=10, candidate_k=100):
    t_total = time.perf_counter()

    t0 = time.perf_counter()
    q_emb = cached_sbert_query(str(query))
    sbert_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    k = min(int(candidate_k), N)
    scores, idx = faiss_index.search(q_emb.reshape(1, -1), k)
    candidate_idx = idx[0]
    sbert_scores = scores[0]
    retrieval_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    X = optimized_runtime_features(
        str(query),
        candidate_idx,
        sbert_scores
    )
    feature_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    pred = np.asarray(
        ranker.predict(X),
        dtype=np.float32
    )
    rank_ms = (time.perf_counter() - t0) * 1000

    order = np.argsort(-pred)[:top_k]
    top_idx = candidate_idx[order]

    result = pd.DataFrame({
        "rank": np.arange(1, len(order)+1),
        "product_id": PID[top_idx],
        "title": TITLE[top_idx],
        "category": CATEGORY[top_idx],
        "brand": BRAND[top_idx],
        "price": RAW_PRICE[top_idx],
        "average_rating": RAW_RATING[top_idx],
        "review_count": RAW_REVIEWS[top_idx],
        "lambdamart_score": pred[order],
        "sbert_retrieval_score": sbert_scores[order],
    })

    total_ms = (time.perf_counter() - t_total) * 1000

    return result, SearchLatency(
        sbert_embedding_ms=sbert_ms,
        faiss_retrieval_ms=retrieval_ms,
        feature_generation_ms=feature_ms,
        lambdamart_ms=rank_ms,
        total_ms=total_ms
    )

print("✅ Optimized search pipeline ready")


In [ ]:

# 9. Integration checks
query_table = (
    w5_dataset[["query_id","query"]]
    .drop_duplicates("query_id")
    .dropna()
    .reset_index(drop=True)
)

query_table["n_tokens"] = (
    query_table["query"]
    .astype(str)
    .map(lambda x: len(tokenize(x)))
)

known_queries = (
    query_table[query_table["n_tokens"].between(2,6)]
    .head(3)["query"]
    .astype(str)
    .tolist()
)

if len(known_queries) < 3:
    known_queries = query_table.head(3)["query"].astype(str).tolist()

test_q = known_queries[0]
clear_query_caches()

q_emb = cached_sbert_query(test_q)
scores, idx = faiss_index.search(q_emb.reshape(1,-1), 100)
X = optimized_runtime_features(test_q, idx[0], scores[0])

checks = {
    "catalog_is_10k_scale": N >= 9000,
    "sbert_catalog_alignment": len(sbert_product_emb) == N,
    "faiss_catalog_alignment": faiss_index.ntotal == N,
    "model_feature_count_match": ranker.num_feature() == len(feature_cols),
    "runtime_feature_order_match": list(X.columns) == list(feature_cols),
    "runtime_feature_count_match": X.shape[1] == ranker.num_feature(),
    "runtime_features_finite": bool(np.isfinite(X.to_numpy()).all()),
    "saved_lambdamart_predicts": bool(np.isfinite(ranker.predict(X)).all()),
}

for k,v in checks.items():
    print(("✅" if v else "❌"), k)

if not all(checks.values()):
    raise AssertionError(
        "Integration check(s) failed: " +
        ", ".join(k for k,v in checks.items() if not v)
    )

print("\n✅ CORE INTEGRATION STILL PASSES AFTER OPTIMIZATION")


In [ ]:

# 10. Quality smoke test
quality_rows = []
quality_latency = []

for q in known_queries:
    clear_query_caches()  # force a fresh-query measurement

    print("\n" + "="*100)
    print("QUERY:", q)

    res, lat = search_products(q, top_k=10, candidate_k=100)
    display(res)

    temp = res.copy()
    temp.insert(0, "query", q)
    quality_rows.append(temp)
    quality_latency.append({"query": q, **asdict(lat)})

quality_results = pd.concat(quality_rows, ignore_index=True)
quality_latency = pd.DataFrame(quality_latency)

display(quality_latency)


In [ ]:

# 11. Fresh-query latency benchmark
bench_queries = known_queries.copy()

for q in query_table.iloc[3:20]["query"].astype(str):
    if q not in bench_queries:
        bench_queries.append(q)

bench_queries = bench_queries[:10]

# Warm model kernels once, but then clear query-level caches.
_ = search_products(bench_queries[0], top_k=10, candidate_k=100)

fresh_rows = []

for q in bench_queries:
    clear_query_caches()

    _, lat = search_products(
        q,
        top_k=10,
        candidate_k=100
    )

    fresh_rows.append({
        "query": q,
        **asdict(lat)
    })

fresh_bench = pd.DataFrame(fresh_rows)
display(fresh_bench)

fresh_summary = (
    fresh_bench
    .drop(columns=["query"])
    .agg(["mean","median","min","max"])
    .T
)
fresh_summary.columns = ["mean_ms","median_ms","min_ms","max_ms"]

display(fresh_summary)

fresh_mean = float(fresh_bench["total_ms"].mean())
fresh_p95 = float(fresh_bench["total_ms"].quantile(0.95))

print(f"Fresh-query mean: {fresh_mean:.2f} ms")
print(f"Fresh-query P95 : {fresh_p95:.2f} ms")
print("Fresh target <200ms:", "✅ PASS" if fresh_mean < 200 else "⚠️ NOT YET")


In [ ]:

# 12. Cached repeat-query benchmark
# Populate query caches once.
clear_query_caches()
for q in bench_queries:
    _ = search_products(q, top_k=10, candidate_k=100)

cached_rows = []

for q in bench_queries:
    _, lat = search_products(
        q,
        top_k=10,
        candidate_k=100
    )
    cached_rows.append({
        "query": q,
        **asdict(lat)
    })

cached_bench = pd.DataFrame(cached_rows)
display(cached_bench)

cached_summary = (
    cached_bench
    .drop(columns=["query"])
    .agg(["mean","median","min","max"])
    .T
)
cached_summary.columns = ["mean_ms","median_ms","min_ms","max_ms"]

display(cached_summary)

cached_mean = float(cached_bench["total_ms"].mean())
cached_p95 = float(cached_bench["total_ms"].quantile(0.95))

print(f"Cached-query mean: {cached_mean:.2f} ms")
print(f"Cached-query P95 : {cached_p95:.2f} ms")
print("Cached target <200ms:", "✅ PASS" if cached_mean < 200 else "⚠️ NOT YET")


In [ ]:

# 13. Latency comparison and bottleneck analysis
comparison = pd.DataFrame([
    {
        "mode": "V1.3 fresh query",
        "mean_total_ms": fresh_mean,
        "p95_total_ms": fresh_p95,
        "target_lt_200_pass": fresh_mean < 200,
    },
    {
        "mode": "V1.3 cached query embeddings",
        "mean_total_ms": cached_mean,
        "p95_total_ms": cached_p95,
        "target_lt_200_pass": cached_mean < 200,
    }
])

display(comparison)

stage_means = pd.DataFrame({
    "stage": [
        "SBERT embedding",
        "FAISS retrieval",
        "Feature generation",
        "LambdaMART"
    ],
    "fresh_mean_ms": [
        fresh_bench["sbert_embedding_ms"].mean(),
        fresh_bench["faiss_retrieval_ms"].mean(),
        fresh_bench["feature_generation_ms"].mean(),
        fresh_bench["lambdamart_ms"].mean(),
    ],
    "cached_mean_ms": [
        cached_bench["sbert_embedding_ms"].mean(),
        cached_bench["faiss_retrieval_ms"].mean(),
        cached_bench["feature_generation_ms"].mean(),
        cached_bench["lambdamart_ms"].mean(),
    ]
})

display(stage_means)


In [ ]:

# 14. Demo query
DEMO_QUERY = "wireless gaming mouse"

clear_query_caches()
demo_results, demo_latency = search_products(
    DEMO_QUERY,
    top_k=10,
    candidate_k=100
)

print("QUERY:", DEMO_QUERY)
display(demo_results)
print(asdict(demo_latency))


In [ ]:

# 15. Save V1.3 outputs
from datetime import datetime

quality_results.to_csv(
    OUT / "week7_v1_3_known_query_ranked_results.csv",
    index=False
)
quality_latency.to_csv(
    OUT / "week7_v1_3_known_query_latency.csv",
    index=False
)
fresh_bench.to_csv(
    OUT / "week7_v1_3_fresh_latency_benchmark.csv",
    index=False
)
cached_bench.to_csv(
    OUT / "week7_v1_3_cached_latency_benchmark.csv",
    index=False
)
fresh_summary.to_csv(
    OUT / "week7_v1_3_fresh_latency_summary.csv"
)
cached_summary.to_csv(
    OUT / "week7_v1_3_cached_latency_summary.csv"
)
comparison.to_csv(
    OUT / "week7_v1_3_latency_comparison.csv",
    index=False
)
stage_means.to_csv(
    OUT / "week7_v1_3_stage_latency_means.csv",
    index=False
)
demo_results.to_csv(
    OUT / "week7_v1_3_demo_ranked_results.csv",
    index=False
)

pd.DataFrame({
    "feature_order": np.arange(1, len(feature_cols)+1),
    "feature": feature_cols
}).to_csv(
    OUT / "week7_v1_3_runtime_feature_schema.csv",
    index=False
)

summary = {
    "week": 7,
    "version": "V1.3",
    "stage": "Latency Optimization",
    "catalog_rows": int(N),
    "candidate_k": 100,
    "top_k": 10,
    "ranker": "Week5 V2 saved LambdaMART",
    "ranker_features": int(len(feature_cols)),
    "clip_runtime_encoder": "CLIPTextModelWithProjection",
    "optimizations": [
        "CLIP text-only query tower",
        "precomputed catalog normalized text/tokens/lengths",
        "precomputed metadata arrays",
        "vectorized CLIP candidate similarity",
        "cached SBERT query embeddings",
        "cached CLIP query embeddings",
    ],
    "fresh_mean_total_ms": fresh_mean,
    "fresh_p95_total_ms": fresh_p95,
    "fresh_target_lt_200_pass": bool(fresh_mean < 200),
    "cached_mean_total_ms": cached_mean,
    "cached_p95_total_ms": cached_p95,
    "cached_target_lt_200_pass": bool(cached_mean < 200),
    "integration_checks": {k: bool(v) for k,v in checks.items()},
    "generated_at_utc": datetime.utcnow().isoformat() + "Z",
}

with open(
    OUT / "week7_v1_3_optimization_summary.json",
    "w"
) as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))


In [ ]:

# 16. Package and download
zip_base = "/content/amazon_week7_v1_3_latency_optimized_results"
zip_path = shutil.make_archive(
    zip_base,
    "zip",
    OUT
)

print("✅ Created:", zip_path)
files.download(zip_path)



## V1.3 decision rule

- If **fresh-query mean < 200 ms**, the original Week 7 latency goal is directly met.
- If fresh queries remain slightly above 200 ms but **cached queries < 200 ms**, report both honestly and carry caching into the FastAPI service.
- Do not change or retrain LambdaMART just to improve latency.
- Once integration still passes and latency is acceptable, proceed to **Week 7 V2 — FastAPI `/search` service**.
